# Bag of Words (BoW)

Convert text messages into numerical features that a machine-learning model can understand.

> **Project:** SMS spam detection  |  **Technique:** Count-based text vectorization

## Learning goals

By the end of this notebook, you will be able to:

- explain how the Bag-of-Words representation works;
- clean and normalize short text messages;
- build a vocabulary with `CountVectorizer`; and
- compare frequency-based and binary BoW features.

### Notebook roadmap
`Setup` → `Load data` → `Preprocess text` → `Build vectors` → `Inspect results`

## 1. Setup

The next cell installs the libraries used in this lesson. In an already configured environment, you can skip it.

**Libraries used:** `pandas` for tabular data, `nltk` for stemming and stop words, and `scikit-learn` for vectorization.

In [ ]:
%pip install pandas nltk scikit-learn

## 2. Import libraries and load the dataset

The dataset contains one SMS message per row. The first column is the target label (`ham` or `spam`) and the second column contains the message text.

> **Path note:** Run this notebook with its folder as the working directory so the relative dataset path resolves correctly.

In [24]:
import pandas as pd

DATA_PATH = 'SpamClassifier-master/smsspamcollection/SMSSpamCollection'
messages = pd.read_csv(
    DATA_PATH,
    sep='\t',
    names=['label', 'message'],
)

messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### Quick data check

Before transforming text, check the dataset size, column types, and class distribution. This helps catch path or parsing issues early.

In [25]:
print(f'Rows: {len(messages):,}')
print(f'Columns: {list(messages.columns)}')
print('\nClass distribution:')
display(messages['label'].value_counts().rename_axis('label').to_frame('count'))

messages.info()

Rows: 5,572
Columns: ['label', 'message']

Class distribution:


,count
label,
ham,4825
spam,747


<class 'pandas.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   label    5572 non-null   str  
 1   message  5572 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB


## 3. Prepare the text

Raw messages contain punctuation, mixed casing, and common words that may add little signal. The preprocessing pipeline below:

1. removes non-alphabetic characters;
2. converts text to lowercase;
3. removes English stop words;
4. applies Porter stemming; and
5. joins the remaining tokens back into a string.

**Important:** Preprocessing is a modeling choice. In a real project, compare alternatives rather than assuming that more cleaning is always better.

In [26]:
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

nltk.download('stopwords', quiet=True)
english_stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [27]:
def clean_message(message: str) -> str:
    """Normalize one message for count-based vectorization."""
    tokens = re.sub(r'[^a-zA-Z]', ' ', message).lower().split()
    tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in english_stop_words
    ]
    return ' '.join(tokens)


corpus = messages['message'].map(clean_message)
messages = messages.assign(clean_message=corpus)

messages[['message', 'clean_message']].head()

,message,clean_message
0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,Free entry in 2 a wkly comp to win FA Cup fina...,free entri wkli comp win fa cup final tkt st m...
3,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


### What changed?

Compare an original message with its cleaned version. Stemming can produce shortened word forms such as `playing` → `play`; this reduces vocabulary size but may make text less readable.

In [28]:
example = messages.loc[0, ['message', 'clean_message']]
print(f"Original: {example['message']}")
print(f"Cleaned : {example['clean_message']}")

Original: Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Cleaned : go jurong point crazi avail bugi n great world la e buffet cine got amor wat


## 4. Build a frequency-based Bag of Words

A **vocabulary** is created from the corpus. Each message then becomes a vector whose columns correspond to vocabulary terms and whose values represent term counts.

The matrix is usually sparse because each message contains only a small fraction of the full vocabulary. `max_features=2500` keeps only the most frequent terms and limits the feature size.

In [29]:
from sklearn.feature_extraction.text import CountVectorizer

MAX_FEATURES = 2_500
count_vectorizer = CountVectorizer(max_features=MAX_FEATURES)
X_count = count_vectorizer.fit_transform(corpus)

print(f'Feature matrix shape: {X_count.shape}')
print(f'Vocabulary size: {len(count_vectorizer.vocabulary_):,}')
print(f'Non-zero values: {X_count.nnz:,}')

Feature matrix shape: (5572, 2500)
Vocabulary size: 2,500
Non-zero values: 40,633


### Read the matrix

- **Rows** represent messages.
- **Columns** represent vocabulary terms.
- **Values** represent how many times a term appears in a message.

The sparse matrix is the preferred representation for larger corpora. Convert only a small sample to a dense DataFrame for inspection.

In [30]:
feature_names = count_vectorizer.get_feature_names_out()
sample_matrix = X_count[:5].toarray()
sample_features = pd.DataFrame(sample_matrix, columns=feature_names)
sample_features.loc[:, sample_features.any(axis=0)].iloc[:, :15]

,alreadi,appli,around,avail,bugi,cine,comp,crazi,cup,dun,earli,entri,fa,final,free
0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,0,0,1,0,1,0,0,2,2,1,1
3,1,0,0,0,0,0,0,0,0,1,1,0,0,0,0
4,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


## 5. Binary Bag of Words

Sometimes the presence of a word matters more than its frequency. With `binary=True`, every non-zero count becomes `1`.

- Frequency BoW: `2` means the term appeared twice.
- Binary BoW: `1` means the term appeared at least once.

This is useful when repeated words should not receive extra weight.

In [31]:
binary_vectorizer = CountVectorizer(
    max_features=MAX_FEATURES,
    binary=True,
)
X_binary = binary_vectorizer.fit_transform(corpus)

print(f'Binary matrix shape: {X_binary.shape}')
print(f'Unique values: {sorted(set(X_binary.data))}')

Binary matrix shape: (5572, 2500)
Unique values: [np.int64(1)]


## 6. Compare the two representations

Both matrices describe the same messages and use the same feature limit. The key difference is the meaning of each non-zero value.

In [32]:
comparison = pd.DataFrame({
    'message': messages['message'].head(5),
    'count_non_zero': X_count[:5].getnnz(axis=1),
    'binary_non_zero': X_binary[:5].getnnz(axis=1),
})
comparison

,message,count_non_zero,binary_non_zero
0,"Go until jurong point, crazy.. Available only ...",11,11
1,Ok lar... Joking wif u oni...,5,5
2,Free entry in 2 a wkly comp to win FA Cup fina...,18,18
3,U dun say so early hor... U c already then say...,5,5
4,"Nah I don't think he goes to usf, he lives aro...",7,7


## Key takeaways

1. Text must be represented numerically before most machine-learning algorithms can use it.
2. A Bag-of-Words vector records vocabulary membership without preserving word order.
3. Frequency and binary features answer slightly different modeling questions.
4. Sparse matrices are more memory-efficient than dense arrays for text data.

### Next step
Use `X_count` or `X_binary` as input features for a classifier, while keeping the labels in `messages['label']`.